In [52]:
print("Installation Dependencies\n")

!pip install -q\
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken
    
print('\Installation completed')

Installation Dependencies



<>:5: SyntaxWarning: invalid escape sequence '\I'
<>:5: SyntaxWarning: invalid escape sequence '\I'
/tmp/ipykernel_58/4183020323.py:5: SyntaxWarning: invalid escape sequence '\I'
  print('\Installation completed')


ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community
\Installation completed


In [4]:
LLM_PROVIDER='groq'
LLM_MODEL='openai/gpt-oss-20b'
CORPUS_PATH = "/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus"

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")
print(f"Corpus path: {CORPUS_PATH}")


Provider: groq
Model: openai/gpt-oss-20b
Corpus path: /kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus


In [5]:
import os
from kaggle_secrets import UserSecretsClient

In [8]:
import os
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("GROQ API KEY is imported!")
else:
    print("GROQ API KEY is not found!")

NameError: name 'load_dotenv' is not defined

In [1]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

print("GROQ_API_KEY loaded from Kaggle secrets.")

ConnectionError: Connection error trying to communicate with service.

In [3]:
import os

print("Key exists:", bool(os.getenv("GROQ_API_KEY")))

Key exists: False


In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "GROQ API KEY
secret_value = UserSecretsClient().get_secret(secret_label)

In [4]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

key = user_secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = key

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

ConnectionError: Connection error trying to communicate with service.

In [5]:
import os

os.environ["GROQ_API_KEY"] = "PASTE_YOUR_GROQ_KEY_HERE"

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Key loaded: True


In [53]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
CORPUS_PATH="./Zyro-dynamics-hr-corpus/" #knowledge base
loader=PyPDFDirectoryLoader(CORPUS_PATH)
document=loader.load()
print(f"Loaded {len(document)}document")

ModuleNotFoundError: No module named 'langchain_community'

In [24]:
import sys
!{sys.executable} -m pip install -U langchain-community langchain-text-splitters

ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community


In [23]:
import importlib.util

print("langchain:", importlib.util.find_spec("langchain"))
print("langchain_community:", importlib.util.find_spec("langchain_community"))
print("langchain_text_splitters:", importlib.util.find_spec("langchain_text_splitters"))

langchain: ModuleSpec(name='langchain', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cf5e8e899d0>, origin='/usr/local/lib/python3.12/dist-packages/langchain/__init__.py', submodule_search_locations=['/usr/local/lib/python3.12/dist-packages/langchain'])
langchain_community: None
langchain_text_splitters: None


In [35]:
import langchain
print("LangChain version:", langchain.__version__)

LangChain version: 1.2.15


In [39]:
import importlib.util

print("pypdf:", importlib.util.find_spec("pypdf"))
print("PyPDF2:", importlib.util.find_spec("PyPDF2"))

pypdf: ModuleSpec(name='pypdf', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cf5e8e8b0e0>, origin='/usr/local/lib/python3.12/dist-packages/pypdf/__init__.py', submodule_search_locations=['/usr/local/lib/python3.12/dist-packages/pypdf'])
PyPDF2: None


In [42]:
import os

folder_path = "/kaggle/input"

print(os.listdir(folder_path))

['notebooks', 'competitions']


In [45]:
import os

print("notebooks:", os.listdir("/kaggle/input/notebooks"))
print("competitions:", os.listdir("/kaggle/input/competitions"))

notebooks: ['pravalikabandela']
competitions: ['project-2-intelligent-rag']


In [47]:
import os

pdf_folder = "/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus"

files = os.listdir(pdf_folder)

print("Total files:", len(files))
print(files)

Total files: 11
['10_Travel_and_Expense_Policy.pdf', '03_Work_From_Home_Policy.pdf', '09_Onboarding_and_Separation_Policy.pdf', '07_IT_and_Data_Security_Policy.pdf', '06_Compensation_and_Benefits_Policy.pdf', '04_Code_of_Conduct.pdf', '01_Employee_Handbook.pdf', '05_Performance_Review_Policy.pdf', '02_Leave_Policy.pdf', '08_Prevention_of_Sexual_Harassment_Policy.pdf', '00_Company_Profile.pdf']


In [48]:
import os
from pypdf import PdfReader

pdf_folder = "/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus"

documents = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(pdf_folder, filename)
        
        reader = PdfReader(file_path)
        text = ""
        
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        
        documents.append({
            "filename": filename,
            "text": text
        })

print("Total PDFs loaded:", len(documents))
print("First PDF:", documents[0]["filename"])
print("Characters:", len(documents[0]["text"]))

Total PDFs loaded: 11
First PDF: 10_Travel_and_Expense_Policy.pdf
Characters: 4771


In [50]:
def split_text(text, chunk_size=1000, chunk_overlap=200):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    
    return chunks


all_chunks = []

for doc in documents:
    chunks = split_text(doc["text"])
    
    for chunk in chunks:
        all_chunks.append({
            "filename": doc["filename"],
            "text": chunk
        })

print("Total chunks:", len(all_chunks))
print("First chunk:")
print(all_chunks[0]["text"][:500])

Total chunks: 88
First chunk:
Zyro Dynamics Pvt. Ltd.
Travel and Expense Policy
Doc Code: ZDL-FIN-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 Navigate the Future
 Travel and Expense Policy
 Document Code
 ZDL-FIN-001
 Version
 V.03
 Effective Date
 01 April 2025
 Document Owner
 Finance and HR Department
INTRODUCTION
Zyro Dynamics requires employees to travel from time to time for client meetings, conferences, inter-office work,
and other business needs. This policy governs all business-related 


In [46]:
import os

folder_path = "/kaggle/input/competitions/project-2-intelligent-rag"

print(os.listdir(folder_path))

['zyro-dynamics-hr-corpus', 'Starter Notebook.ipynb', 'sample submission.csv', 'test.csv']


In [44]:
import os

folder_path = "/kaggle/input/notebook-competition"

print(os.listdir(folder_path))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/notebook-competition'

In [7]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("LangChain imports working!")

ModuleNotFoundError: No module named 'langchain_community'

In [8]:
!pip install -q langchain-community langchain-text-splitters

ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community


In [12]:
import langchain
print(langchain.__version__)

1.2.15


In [13]:
import sys
!{sys.executable} -m pip install -U langchain-community langchain-text-splitters

ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community


In [11]:
!pip install -q langchain-community langchain-text-splitters

ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community


In [10]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks=splitter.split_documents(document)
print(f"created{len(chunks)}chunks")

NameError: name 'RecursiveCharacterTextSplitter' is not defined

## Embeddings
HuggingFaceEmbeddings

In [54]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
print("Embeddings done!")

 


ModuleNotFoundError: No module named 'langchain_huggingface'

VectorDB + Retrival

FAISS Vector DB

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)

#Retrival
retrieval = vectorstore.as_retriever(search_kwargs ={"k": 3 })
print("Done!")

## LLM Initialization
LLM model - groq


In [7]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0.7,
    max_tokens = 500
)
print("LLM Model initialized")

ModuleNotFoundError: No module named 'langchain_groq'

## RAG CHAIN

Initialization

In [ ]:
def rag_chain(question):

    docs = retrieval.invoke(question)
    context = format_docs(docs)

    chain = RAG_PROMPT | llm_model | StrOutputParser()

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    return {"answer": answer}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template("""
Answer the question using only the context provided.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm_model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY")
)

print("LLM initialized!")

In [ ]:
print("Model:", llm_model.model_name)

## Guardials
(To ensure no context is used beyond the corups)

GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question
below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct, or other internal HR topics) or something
out of scope (general knowledge, coding help, unrelated small talk, etc.).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question
below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct, or other internal HR topics) or something
out of scope (general knowledge, coding help, unrelated small talk, etc.).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)
def ask_bot(question: str):

    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()

    verdict = guardial_chain.invoke(f"question : {question}").strip().upper()

    if 'OUT_OF_SCOPE' in verdict:
        return {'answer': REFUSAL_MESSAGE, "sources": []}

    return rag_chain(question)

print("Guardial Initialized!")

In [ ]:
test_questions = [
    "How many casual leaves do I get per year?",
    "What is the internet reimbursement limit?",
    "What's the capital of France?",
]
for i, q in enumerate(test_questions, 1):
    result = ask_bot(q)

    print(f"Q{i} : {q}")
    print(f"A{i} : {result['answer']}")
    sources=result.get("sources",[])
    if sources:
       print(
            "Sources: ",
            [d.metadata.get('source') for d in sources]
        )
    print("-" * 60)

In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv")
test_df
answers = []

for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    
    print(f"{i+1}/{len(test_df)} {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers

submission = test_df[["question_id", "answer"]]

submission.to_csv("submission.csv", index=False)

submission.head()